# RhoFold+ — Google Colab Verification & Custom Target Notebook

**Project:** RNA 3D Structure Prediction for Template-Sparse RNA Sequences  
**Model:** RhoFold+  
**Official repository:** `ml4bio/RhoFold`

This notebook is Colab-first and contains:

1. bundled verification using `3owzA.fasta + 3owzA.a3m`
2. custom target with precomputed A3M — recommended for the FYP benchmark
3. custom sequence-only mode — quick-reference / ablation only

The full automatic-MSA databases are intentionally skipped because the official workflow requires RNAcentral, Rfam and nt databases totaling roughly 900 GB.

Expected bundled outputs:
`results.npz`, `ss.ct`, `unrelaxed_model.pdb`, `relaxed_1000_model.pdb`, and `log.txt`.

## 1. GPU and host check

In [ ]:
import os, sys, time, threading, subprocess, platform, re, csv, shutil
from pathlib import Path
print("Host Python:", sys.version)
print("Platform:", platform.platform())
subprocess.run(["nvidia-smi"], check=False)

## 2. Clone the official RhoFold+ repository

In [ ]:
%cd /content
if not Path("/content/RhoFold").exists():
    !git clone https://github.com/ml4bio/RhoFold.git
else:
    print("RhoFold repository already exists; reusing it.")

%cd /content/RhoFold
commit = subprocess.check_output(["git","rev-parse","HEAD"], text=True).strip()
print("RhoFold commit:", commit)

## 3. Inspect the official Linux environment

In [ ]:
%cd /content/RhoFold
!sed -n '1,240p' envs/environment_linux.yaml

## 4. Install Micromamba

In [ ]:
%cd /content
if not Path("/content/bin/micromamba").exists():
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj
!/content/bin/micromamba --version

## 5. Create the official RhoFold environment

In [ ]:
%cd /content/RhoFold
envs = subprocess.run(
    ["/content/bin/micromamba","-r","/content/micromamba_root","env","list"],
    capture_output=True, text=True
)
if "rhofold" not in envs.stdout:
    subprocess.run([
        "/content/bin/micromamba","-r","/content/micromamba_root",
        "create","-y","-n","rhofold",
        "-f","/content/RhoFold/envs/environment_linux.yaml"
    ], check=True)
else:
    print("rhofold environment already exists.")

## 6. Install RhoFold package

In [ ]:
%cd /content/RhoFold
proc_install = subprocess.run([
    "/content/bin/micromamba","-r","/content/micromamba_root",
    "run","-n","rhofold","python","setup.py","install"
], cwd="/content/RhoFold", capture_output=True, text=True)
print(proc_install.stdout)
print(proc_install.stderr)
if proc_install.returncode != 0:
    raise RuntimeError("RhoFold package installation failed.")

## 7. Verify RhoFold environment / CUDA

In [ ]:
verify_code = """
import sys, numpy, torch
print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
"""
verify = subprocess.run([
    "/content/bin/micromamba","-r","/content/micromamba_root",
    "run","-n","rhofold","python","-c",verify_code
], capture_output=True, text=True)
print("RETURN CODE:", verify.returncode)
print("STDOUT:\n", verify.stdout)
print("STDERR:\n", verify.stderr)
if verify.returncode != 0:
    raise RuntimeError("Environment verification failed.")

## 8. Download the official pretrained checkpoint

In [ ]:
%cd /content/RhoFold
ckpt_dir = Path("/content/RhoFold/pretrained")
ckpt_dir.mkdir(parents=True, exist_ok=True)
ckpt = ckpt_dir / "RhoFold_pretrained.pt"
if not ckpt.exists():
    !wget https://huggingface.co/cuhkaih/rhofold/resolve/main/rhofold_pretrained_params.pt -O /content/RhoFold/pretrained/RhoFold_pretrained.pt
if not ckpt.exists() or ckpt.stat().st_size == 0:
    raise FileNotFoundError("RhoFold checkpoint download failed.")
print("Checkpoint:", ckpt)
print("Checkpoint size (MB):", round(ckpt.stat().st_size/(1024**2),2))

## 9. Inspect the bundled verification input

In [ ]:
%cd /content/RhoFold
example_dir = Path("/content/RhoFold/example/input/3owzA")
for p in sorted(example_dir.glob("*")):
    print(p)
required = [example_dir/"3owzA.fasta", example_dir/"3owzA.a3m"]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing bundled RhoFold inputs: " + ", ".join(missing))
print("Bundled FASTA and A3M are present.")

## 10. Check RhoFold inference CLI

In [ ]:
%cd /content/RhoFold
h = subprocess.run([
    "/content/bin/micromamba","-r","/content/micromamba_root",
    "run","-n","rhofold","python","inference.py","-h"
], cwd="/content/RhoFold", capture_output=True, text=True)
print(h.stdout)
print(h.stderr)
if h.returncode != 0:
    raise RuntimeError("Could not load inference.py")

## 11. Run bundled RhoFold+ verification with precomputed A3M

In [ ]:
%cd /content/RhoFold

verify_output = Path("/content/RhoFold/example/output/3owzA_verify")
if verify_output.exists():
    shutil.rmtree(verify_output)
verify_output.mkdir(parents=True, exist_ok=True)

gpu_samples = []
stop_monitor = False

def monitor_gpu():
    global stop_monitor
    while not stop_monitor:
        try:
            out = subprocess.check_output(
                ["nvidia-smi","--query-gpu=memory.used","--format=csv,noheader,nounits"],
                text=True
            ).strip().splitlines()
            if out:
                gpu_samples.append(int(out[0]))
        except Exception:
            pass
        time.sleep(0.5)

monitor_thread = threading.Thread(target=monitor_gpu, daemon=True)
monitor_thread.start()
start = time.perf_counter()

proc = subprocess.run([
    "/content/bin/micromamba","-r","/content/micromamba_root",
    "run","-n","rhofold","python","inference.py",
    "--input_fas","./example/input/3owzA/3owzA.fasta",
    "--input_a3m","./example/input/3owzA/3owzA.a3m",
    "--output_dir",str(verify_output),
    "--ckpt","./pretrained/RhoFold_pretrained.pt",
    "--device","cuda:0"
], cwd="/content/RhoFold", capture_output=True, text=True)

elapsed = time.perf_counter() - start
stop_monitor = True
monitor_thread.join(timeout=2)
peak_gpu_mb = max(gpu_samples) if gpu_samples else None

print("===== STDOUT =====")
print(proc.stdout)
print("===== STDERR =====")
print(proc.stderr)
print("=== RUN SUMMARY ===")
print("Exit code:", proc.returncode)
print(f"Runtime: {elapsed:.2f} s")
print("Peak observed GPU memory:", f"{peak_gpu_mb} MiB" if peak_gpu_mb is not None else "Unavailable")

if proc.returncode != 0:
    raise RuntimeError("RhoFold inference failed. Check STDERR above.")

## 12. Verify bundled output files

In [ ]:
expected_relaxed = verify_output / "relaxed_1000_model.pdb"
expected_unrelaxed = verify_output / "unrelaxed_model.pdb"

print("Relaxed PDB:", expected_relaxed)
print("Exists:", expected_relaxed.exists())

for p in sorted(verify_output.glob("*")):
    print(p)

if not expected_relaxed.exists() and not expected_unrelaxed.exists():
    raise FileNotFoundError("Neither relaxed nor unrelaxed RhoFold PDB was generated.")

primary_pdb = expected_relaxed if expected_relaxed.exists() else expected_unrelaxed
print("Primary verified PDB:", primary_pdb)

## 13. Save bundled-run reproducibility metadata

In [ ]:
version_code = """
import sys, numpy, torch
print(sys.version.split()[0])
print(numpy.__version__)
print(torch.__version__)
"""
versions = subprocess.check_output([
    "/content/bin/micromamba","-r","/content/micromamba_root",
    "run","-n","rhofold","python","-c",version_code
], text=True).strip().splitlines()

gpu_name = subprocess.check_output(
    ["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
    text=True
).strip().splitlines()[0]

record = {
    "model":"RhoFold+",
    "model_commit":commit,
    "checkpoint":"RhoFold_pretrained.pt",
    "target":"3owzA",
    "input_fasta":"example/input/3owzA/3owzA.fasta",
    "input_msa":"example/input/3owzA/3owzA.a3m",
    "mode":"precomputed_msa",
    "output":str(primary_pdb),
    "python":versions[0] if len(versions)>0 else "",
    "numpy":versions[1] if len(versions)>1 else "",
    "pytorch":versions[2] if len(versions)>2 else "",
    "gpu":gpu_name,
    "runtime_seconds":round(elapsed,3),
    "peak_gpu_memory_mib_observed":peak_gpu_mb,
    "exit_code":proc.returncode
}

bundled_log = Path("/content/rhofold_run_log.csv")
with bundled_log.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=record.keys())
    w.writeheader()
    w.writerow(record)
print(record)
print("Saved:", bundled_log)

## 14. Optional — save bundled verification outputs to Google Drive

In [ ]:
# OPTIONAL CELL
# from google.colab import drive
# drive.mount("/content/drive")
# destination = Path("/content/drive/MyDrive/RNA3D_Benchmark/RhoFold")
# destination.mkdir(parents=True, exist_ok=True)
# shutil.copytree(verify_output, destination/"3owzA_verify", dirs_exist_ok=True)
# shutil.copy2(bundled_log, destination/bundled_log.name)
# print("Saved to:", destination)

# Custom Target — Precomputed A3M Mode

This is the recommended FYP benchmark mode.

Required:

```text
TARGET.fasta
TARGET.a3m
```

The A3M should be produced using the standardized team MSA policy.

## 15. Enter your RNA sequence

In [ ]:
JOB_NAME = "my_rna"

RNA_SEQUENCE = """
GGGAAACCCUUUGGGAAACCC
""".replace("\n","").replace(" ","").upper()

if not re.fullmatch(r"[ACGU]+", RNA_SEQUENCE):
    raise ValueError("RNA sequence must contain only A, C, G and U.")

custom_root = Path("/content/RhoFold/custom_input")
custom_target = custom_root / JOB_NAME
custom_target.mkdir(parents=True, exist_ok=True)

custom_fasta = custom_target / f"{JOB_NAME}.fasta"
custom_fasta.write_text(f">{JOB_NAME}\n{RNA_SEQUENCE}\n")

print("Target:", JOB_NAME)
print("Length:", len(RNA_SEQUENCE))
print("FASTA:", custom_fasta)

## 16. Upload your precomputed A3M

In [ ]:
from google.colab import files
uploaded = files.upload()
a3m_files = [name for name in uploaded if name.lower().endswith(".a3m")]
if len(a3m_files) != 1:
    raise ValueError("Please upload exactly one .a3m file.")
custom_a3m = custom_target / f"{JOB_NAME}.a3m"
shutil.move(a3m_files[0], custom_a3m)
print("A3M saved:", custom_a3m)

## 17. Run RhoFold+ on custom FASTA + A3M

In [ ]:
%cd /content/RhoFold
custom_output = Path("/content/RhoFold/custom_output") / JOB_NAME
if custom_output.exists():
    shutil.rmtree(custom_output)
custom_output.mkdir(parents=True, exist_ok=True)

custom_proc = subprocess.run([
    "/content/bin/micromamba","-r","/content/micromamba_root",
    "run","-n","rhofold","python","inference.py",
    "--input_fas",str(custom_fasta),
    "--input_a3m",str(custom_a3m),
    "--output_dir",str(custom_output),
    "--ckpt","./pretrained/RhoFold_pretrained.pt",
    "--device","cuda:0"
], cwd="/content/RhoFold", capture_output=True, text=True)

print(custom_proc.stdout)
print(custom_proc.stderr)
if custom_proc.returncode != 0:
    raise RuntimeError("Custom RhoFold inference failed.")

## 18. Verify custom A3M-mode outputs

In [ ]:
custom_relaxed = custom_output / "relaxed_1000_model.pdb"
custom_unrelaxed = custom_output / "unrelaxed_model.pdb"
for p in sorted(custom_output.glob("*")):
    print(p)
if not custom_relaxed.exists() and not custom_unrelaxed.exists():
    raise FileNotFoundError("No custom RhoFold PDB was generated.")
custom_primary_pdb = custom_relaxed if custom_relaxed.exists() else custom_unrelaxed
print("Primary custom PDB:", custom_primary_pdb)

# Optional Custom Target — Sequence-Only Mode

The official README states this mode is still in testing and is less accurate than the MSA version. Use it only for quick reference, ablation, or debugging.

## 19. Run custom sequence-only RhoFold+ prediction

In [ ]:
%cd /content/RhoFold
single_output = Path("/content/RhoFold/custom_output") / f"{JOB_NAME}_single_seq"
if single_output.exists():
    shutil.rmtree(single_output)
single_output.mkdir(parents=True, exist_ok=True)

single_proc = subprocess.run([
    "/content/bin/micromamba","-r","/content/micromamba_root",
    "run","-n","rhofold","python","inference.py",
    "--input_fas",str(custom_fasta),
    "--single_seq_pred","True",
    "--output_dir",str(single_output),
    "--ckpt","./pretrained/RhoFold_pretrained.pt",
    "--device","cuda:0"
], cwd="/content/RhoFold", capture_output=True, text=True)

print(single_proc.stdout)
print(single_proc.stderr)
if single_proc.returncode != 0:
    raise RuntimeError("Sequence-only RhoFold run failed.")

## 20. Benchmark policy for this FYP

Primary benchmark:

```text
FASTA + standardized/precomputed A3M
            ↓
         RhoFold+
            ↓
relaxed_1000_model.pdb
```

Rules:
- use one standardized MSA-generation policy across benchmark targets
- record MSA depth/quality separately from structural-template scarcity
- do not use native 3D coordinates during inference
- native structure is evaluation-only
- record commit, checkpoint, runtime, GPU and peak observed VRAM
- sequence-only mode is a separate ablation/quick-reference condition
- do not require each team member to download the ~900 GB automatic-MSA databases